## Carregamento da base de dados de exemplo

*Kalil Nege — RA 2409020 · OPT098 - Tópicos Especiais em Sistemas de Computação (Prof. Dr. Rodrigo Hübner)*

Neste conjunto de notebooks aplico o roteiro de extração de características do repositório `bneurd/tutoriais` (`SomeFeatures`) ao tema da minha revisão: **SSVEP e ritmos cerebrais (beta) como indicadores de atenção**.

Será utilizado o **AVI SSVEP Dataset** (Vilic, 2013 — www.setzner.com), o mesmo empregado nos notebooks `SSVEP/` do repositório. Os pontos importantes da base de dados são:

- **Um único canal** de EEG (`Oz`), referência em `Fz` e terra em `Fpz` (sistema 10-20);
- Equipamento `g.USBamp`, taxa de amostragem de **512 Hz**, filtro filtro rejeita-faixa analógico em 50 Hz;
- Estímulos em monitor LCD BenQ XL2420T (120 Hz), a 60 cm do participante;
- **Single target**: 4 participantes, ensaios de **30 segundos** olhando para um único alvo cintilante;
- **Multi target**: 5 participantes, 2 sessões, 10 ensaios de **16 segundos** com 7 alvos na tela ao mesmo tempo (6, 6.5, 7, 7.5, 8.2, 9.3 e 10 Hz). O participante precisa **escolher (atenção seletiva)** em qual alvo focar.

> Para executar, copie a pasta `SSVEP/datasets/avi` do repositório `tutoriais` para `dados/avi` (ao lado destes notebooks).

In [1]:
# Carregamento de um arquivo do participante 1 (single target)

from scipy.io import loadmat
import numpy as np

mat = loadmat('dados/avi/single/Sub1_singletarget.mat')
mat.keys()

dict_keys(['__header__', '__version__', '__globals__', 'Data'])

In [2]:
# Apresentando informações básicas presentes no dataset

registro = mat['Data'][0][0]
print('Campos:', registro.dtype.names)

fs = int(registro['AmpSamplingFrequency'][0][0])
print(f'\nTaxa de amostragem: {fs} Hz')
print('Eletrodo:', registro['ElectrodePlacement'][0])
print('Formato do EEG (amostras, ensaios):', registro['EEG'].shape)
print('Frequências alvo de cada ensaio:', registro['TargetFrequency'].ravel())

Campos: ('EEG', 'TargetFrequency', 'RecordedDate', 'AmpSamplingFrequency', 'TargetSize', 'SubjectDistance', 'SubjectAge', 'SubjectGender', 'StimulusDevice', 'ElectrodePlacement')

Taxa de amostragem: 512 Hz
Eletrodo: Oz
Formato do EEG (amostras, ensaios): (15360, 27)
Frequências alvo de cada ensaio: [10.  10.  10.  12.  12.  12.   6.5  6.5  6.5  6.   6.   6.   6.   6.
  6.   7.5  7.5  7.5  7.   7.   7.   8.2  8.2  8.2  9.3  9.3  9.3]


### Organização das informações da base de dados

Os arquivos *single* e *multi target* possuem os campos em ordens diferentes, então o acesso é feito **pelo nome do campo** (`registro['EEG']`, `registro['TargetFrequency']`).

O participante 1 do *single target* possui ensaios em 12 Hz, frequência que não existe nos demais. Assim como no notebook `load_single_avi.ipynb`, esses ensaios são descartados, ficando as **7 frequências comuns**.

Além do sinal e do rótulo, guardo para cada ensaio o **participante** e um **identificador do ensaio**. Isso será essencial na validação: janelas de um mesmo ensaio não podem ficar ao mesmo tempo no treino e no teste.

In [3]:
FREQS = [6.0, 6.5, 7.0, 7.5, 8.2, 9.3, 10.0]


def carrega_avi(arquivos):
    """Retorna (X, y, participante) com um ensaio por linha."""
    X, y, suj = list(), list(), list()
    for participante, arquivo in arquivos:
        d = loadmat(arquivo)['Data'][0][0]
        eeg = d['EEG']
        alvos = d['TargetFrequency'].ravel()
        for c in range(eeg.shape[1]):
            alvo = round(float(alvos[c]), 1)
            if alvo not in FREQS:
                continue
            X.append(eeg[:, c])
            y.append(alvo)
            suj.append(participante)
    return np.array(X), np.array(y), np.array(suj)


single = [(i, f'dados/avi/single/Sub{i}_singletarget.mat') for i in range(1, 5)]
multi = [(i, f'dados/avi/multi/Sub{i}_{s}_multitarget.mat') for i in range(1, 6) for s in (1, 2)]

X_single, y_single, suj_single = carrega_avi(single)
X_multi, y_multi, suj_multi = carrega_avi(multi)

print('Single target:', X_single.shape, '-> 30 s x 512 Hz =', 30 * 512)
print('Multi target: ', X_multi.shape, '-> 16 s x 512 Hz =', 16 * 512)

Single target: (92, 15360) -> 30 s x 512 Hz = 15360
Multi target:  (100, 8192) -> 16 s x 512 Hz = 8192


In [4]:
# Quantidade de ensaios por frequência (verificação de balanceamento)

for nome, y in (('single', y_single), ('multi', y_multi)):
    freqs, qtd = np.unique(y, return_counts=True)
    print(nome, ' | '.join(f'{f} Hz: {q}' for f, q in zip(freqs, qtd)))

single 6.0 Hz: 15 | 6.5 Hz: 13 | 7.0 Hz: 15 | 7.5 Hz: 12 | 8.2 Hz: 13 | 9.3 Hz: 12 | 10.0 Hz: 12
multi 6.0 Hz: 40 | 6.5 Hz: 5 | 7.0 Hz: 10 | 7.5 Hz: 15 | 8.2 Hz: 25 | 9.3 Hz: 5


Observações:

- No *single target* as classes estão aproximadamente balanceadas.
- No *multi target* **não existe ensaio em 10 Hz**, e a distribuição é desbalanceada (o alvo de 6 Hz aparece bem mais). Por isso, além da acurácia, será reportada a **acurácia balanceada**.

In [5]:
# salvamento dos dados em numpy arrays

np.save('dados/single_x.npy', X_single)
np.save('dados/single_y.npy', y_single)
np.save('dados/single_sub.npy', suj_single)

np.save('dados/multi_x.npy', X_multi)
np.save('dados/multi_y.npy', y_multi)
np.save('dados/multi_sub.npy', suj_multi)